# RealEstate Price Model Training

Loads data from data.jsonl or data.json, cleans it, and trains a baseline model to predict price.

In [ ]:
import os, io, re, html
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


## Load Data
Attempts from `/content` or Google Drive. If not found, prompts for file upload.

In [ ]:
if IN_COLAB:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

candidates = [
    '/content/data.jsonl',
    '/content/data.json',
    '/content/drive/MyDrive/data.jsonl',
    '/content/drive/MyDrive/data.json'
]

df = None
data_path = None
for p in candidates:
    if os.path.exists(p):
        if p.endswith('.jsonl'):
            df = pd.read_json(p, lines=True)
        else:
            df = pd.read_json(p)
        data_path = p
        break

if df is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded')
        name = next(iter(uploaded))
        bio = io.BytesIO(uploaded[name])
        if name.endswith('.jsonl'):
            df = pd.read_json(bio, lines=True)
        else:
            df = pd.read_json(bio)
        data_path = name
    except Exception as e:
        raise RuntimeError('Place data.jsonl or data.json in /content or Drive, or upload it.') from e

df.head(3)

## Clean and Engineer Features
- Convert `size` to area in m².
- Convert GPS DMS strings to decimal latitude and longitude.
- Prepare features: `propertyType` (one-hot), `area`, `lat`, `lon`.

In [ ]:
def parse_area(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.nan
    t = html.unescape(str(s))
    t = t.replace('\u00A0', ' ')
    m = re.search(r"\d[\d\s\u00A0]*", t)
    if not m:
        return np.nan
    num = re.sub(r"[\s\u00A0]", "", m.group(0))
    try:
        return float(num)
    except Exception:
        return np.nan

def dms_to_decimal(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    t = html.unescape(str(v))
    nums = re.findall(r"\d+(?:\.\d+)?", t)
    hemi = None
    hm = re.search(r"[NSEW]", t)
    if hm:
        hemi = hm.group(0)
    if not nums:
        return np.nan
    deg = float(nums[0])
    minutes = float(nums[1]) if len(nums) > 1 else 0.0
    seconds = float(nums[2]) if len(nums) > 2 else 0.0
    dec = deg + minutes / 60.0 + seconds / 3600.0
    if hemi in ('S', 'W'):
        dec = -dec
    return dec

df['area'] = df['size'].apply(parse_area)
df['lat'] = df['gpsX'].apply(dms_to_decimal)
df['lon'] = df['gpsY'].apply(dms_to_decimal)

df_model = df[['propertyType', 'area', 'lat', 'lon', 'price']].dropna()
df_model = df_model[(df_model['area'] > 0) & (df_model['price'] > 0)]
df_model.head(3)

,propertyType,area,lat,lon,price
0,Domy a vily,123.0,50.103944,13.737028,7490000
1,Byty,160.0,50.145944,14.339167,12990000
2,Byty,83.0,49.743111,13.379139,6250000


## Train/Test Split and Model Training
Uses RandomForestRegressor as a strong baseline.

In [ ]:
X = df_model[['propertyType', 'area', 'lat', 'lon']]
y = df_model['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pre = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['propertyType']),
        ('num', 'passthrough', ['area', 'lat', 'lon'])
    ]
)

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
pipe = Pipeline(steps=[('pre', pre), ('model', rf)])

pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)
rmse, r2

(np.float64(6039222.394037551), 0.49625602019630766)

## Save Model Artifact
Saves `model.joblib` to `/content` and to Drive if mounted.

In [ ]:
model_local = '/content/model.joblib'
joblib.dump(pipe, model_local)
print(f"Model saved to: {model_local}")

Model saved to: /content/model.joblib
